In [1]:
import ee 
import geemap
ee.Authenticate
ee.Initialize

<function ee.Initialize(credentials: Any | None = 'persistent', url: str | None = None, cloud_api_key: str | None = None, http_transport: Any | None = None, project: str | int | None = None) -> None>

In [2]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

In [3]:
districts = ee.FeatureCollection("projects/deforestation-panama/assets/Districts")

In [4]:
roads = ee.FeatureCollection("projects/deforestation-495419/assets/Panama_OSM_Roads")

In [ ]:
# Function to calculate road density for each district
def calculate_road_density(district):

    # Get district geometry
    district_geom = district.geometry()

    # Calculate district area in square kilometers
    district_area = district_geom.area(maxError=1).divide(1e6)

    # Clip roads to district boundaries
    district_roads = roads.filterBounds(district_geom)

    # Calculate total road length within district (meters)
    road_length = district_roads.map(
        lambda road: road.set(
            "length",
            road.geometry().intersection(
                district_geom, maxError=1
            ).length(maxError=1)
        )
    ).aggregate_sum("length")

    # Convert road length to kilometers
    road_length_km = ee.Number(road_length).divide(1000)

    # Calculate road density (km/km2)
    road_density = road_length_km.divide(district_area)

    # Add results to district feature
    return district.set({
        "road_length_km": road_length_km,
        "area_km2": district_area,
        "road_density": road_density
    })


# Apply function to all districts
districts_with_density = districts.map(calculate_road_density)


# Select district name and road density
# IMPORTANT: Replace 'DISTRICT_NAME' with the actual
# district-name field in your shapefile if necessary.

results = districts_with_density.select([
    "DISTRICT_NAME",
    "road_density"
])


# Convert Earth Engine results to a Pandas DataFrame
data = results.getInfo()

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "District": feature["properties"]["DISTRICT_NAME"],
        "Road_Density_km_per_km2": feature["properties"]["road_density"]
    }
    for feature in data["features"]
])

# Display dataframe
print(df)

In [ ]:
# Export dataframe to CSV
df.to_csv("Panama_District_Road_Density.csv", index=False)

print("CSV successfully exported!")